In [ ]:
from autogluon.tabular import TabularPredictor
from augmentation.join_selection import JoinSelection
from augmentation.utils.config import DiscoveryConfig
from experiments.base_tables.base_table_preprocessing import PreProcessor
import polars as pl
import numpy as np
import yaml

In [ ]:
base_table_path = 'experiments/base_tables/inspections_classification/inspections_classification.csv'
base_table_splits_path = 'experiments/base_tables/inspections_classification/splits.json'
preprocessor = PreProcessor(base_table_path, base_table_splits_path, 2)
X, query_col, target, nan_mask = preprocessor.run()
X.write_csv('experiments/base_tables/inspections_classification/inspections_classification_preprocessed.csv')
X.head()

In [ ]:
data = X.to_pandas()
train_data = data.sample(frac=0.8, random_state=42)
test_data = data.drop(train_data.index)
predictor = TabularPredictor(label=target, problem_type='multiclass').fit(train_data)
predictions = predictor.predict(test_data)
performance = predictor.evaluate(test_data)

In [ ]:
performance

In [ ]:
import pandas as pd
data = pd.read_csv('experiments/base_tables/inspections_classification/backward.csv')
train_data = data.sample(frac=0.8, random_state=42)
test_data = data.drop(train_data.index)
predictor = TabularPredictor(label=target, problem_type='multiclass').fit(train_data)
predictions = predictor.predict(test_data)
performance = predictor.evaluate(test_data)

In [ ]:
performance

In [ ]:
user_table_agg = pl.read_parquet('user_table_agg.parquet')

In [ ]:
user_table_agg

In [ ]:
from augmentation.utils.common import compute_outer_products_adaptive as compute_outer_products

In [ ]:
cofactors, _ = compute_outer_products(np.array(user_table_agg.select('sum').to_series().to_list()))

In [ ]:
np.array(user_table_agg.select('sum').to_series().to_list())

In [ ]:
import yaml

with open('experiments/downstream/config.yml', 'r') as f:
    config = yaml.safe_load(f)

config

In [ ]:
overlap_query_results

In [ ]:
import polars as pl
from experiments.downstream.experiment_executor import ExperimentExecutor
from augmentation.join_selection import JoinSelection
from augmentation.utils.config import DiscoveryConfig
df = pl.read_csv('experiments/downstream/experiments.csv')

In [ ]:
import inspect

def validate_kwargs(method, kwargs):
    sig = inspect.signature(method)
    try:
        sig.bind(**kwargs)
        return True
    except TypeError as e:
        return False, str(e)

for lake, config in df.group_by('lake'):
    headers = config.columns
    for row in config.iter_rows():
        exp_config = {k: row[i] for i, k in enumerate(headers)}
        lake = exp_config.pop('lake')
        base_table_name = exp_config.pop('table')
        strategy = exp_config.pop('algorithm')
        strat_index = headers.index('strategy')
        strat = row[strat_index]
        execution_data = ExperimentExecutor.from_params(exp_config, strat)
        worker = JoinSelection(**execution_data.init_args)
        # config = DiscoveryConfig(**execution_data.discovery_config_args)
        # find_best_joins_kwargs = {'user_table_processed': df, 'top_k': 50, 'n_jobs': 64, 'config': DiscoveryConfig}
        # find_best_joins_kwargs.update(execution_data.run_args)
        # print(validate_kwargs(worker.find_best_joins, find_best_joins_kwargs))

In [ ]:
execution_data = ExperimentExecutor.from_params(exp_config, strat)

In [ ]:
execution_data.init_args

In [ ]:
execution_data.run_args

In [ ]:
execution_data.discovery_config_args

In [ ]:
execution_data.unknown_args

In [ ]:
join_selection_query_results = pl.read_parquet('join_selection_query_results.parquet')

In [ ]:
for group in join_selection_query_results.group_by(['table_index', 'feature_index', 'table_column_index']):
    lens = group[1].select(pl.col('sum').list.len().unique())
    if lens.shape[0] > 1:
        print(group[1])

In [ ]:
df = pl.read_csv('experiments/downstream/experiments.csv')
import yaml
with open('experiments/downstream/config.yml', 'r') as f:
    config = yaml.safe_load(f)
lake = 'nyc'
table = 'crime'
base_tables = config['lakes'][lake]['base_tables']
for base_table in base_tables:
    for key in base_table:
        if key == table:
            task = base_table[key][0]['task']

In [ ]:
import polars as pl

base = pl.read_csv('experiments/base_tables/base_scores.csv')
base = base.with_columns(pl.col('').alias('table'))
aug = pl.read_csv('experiments/downstream/logs/all_results.csv')
aug = (
    aug
    .with_columns(pl.col('').str.split('_').list.to_struct(upper_bound=3, fields=['lake', 'table', 'algorithm']).struct.unnest())
    .select(['table', 'algorithm', pl.col('root_mean_squared_error').mul(-1), 'f1'])
)
res = aug.join(base.select(['table', pl.col('rmse').alias('root_mean_squared_error'), pl.col('f1_weighted').alias('f1')]), on=['table'], how='left')

In [6]:
import polars as pl
df = pl.read_csv('/mnt/data1/lakes/canada_us_uk_open_data/extracted/CAN_CSV0000000000002119.csv', ignore_errors=True, separator=',')

In [7]:
df

ï»¿year,month,request_number,summary_en,summary_fr,disposition,pages,umd_number,owner_org,owner_org_title
i64,i64,str,str,str,str,i64,f64,str,str
2020,6,"""ACA-2020-00220""","""1. Technical Standard Developm…","""1. Technical Standard Developm…","""DP""",45,null,"""casdo-ocena""","""Accessibility Standards Canada…"
2020,8,"""ACA-2020-00255""","""1. Technical Committee_Plain L…","""1. Technical Committee_Plain L…","""DA""",16,null,"""casdo-ocena""","""Accessibility Standards Canada…"
2020,7,"""ACA-2020-00427""","""Send copies of the current art…","""Veuillez envoyer des copies de…","""NE""",0,null,"""casdo-ocena""","""Accessibility Standards Canada…"
2020,7,"""ACA-2020-00441""","""Send documents that provide ob…","""Veuillez envoyer des documents…","""NE""",0,null,"""casdo-ocena""","""Accessibility Standards Canada…"
2020,8,"""ACA-2020-00533""","""Provide documentary evidence t…","""Fournir la preuve documentaire…","""NE""",0,null,"""casdo-ocena""","""Accessibility Standards Canada…"
…,…,…,…,…,…,…,…,…,…
2019,6,"""2019-02""","""Provide all documents and reco…","""Tous les documents et dossiers…","""DA""",696,589.0,"""yesab-oeesy""","""Yukon Environmental and Socio-…"
2021,2,"""2021-002""","""All travel and expenses for st…","""Tous les dÃ©placements et dÃ©p…","""DP""",670,589.0,"""yesab-oeesy""","""Yukon Environmental and Socio-…"
2021,1,"""2021-001""","""Request for all internal corre…","""Demande de toute correspondanc…","""EC""",5000000,589.0,"""yesab-oeesy""","""Yukon Environmental and Socio-…"


In [ ]:
from augmentation.index import ExhaustiveIndex

worker = ExhaustiveIndex()
worker._prune_features(df.select(pl.nth(1)).lazy())

In [77]:
base_scores = pl.read_csv('experiments/base_tables/base_simple_scores.csv').rename({'': 'table'})
base_scores.filter(pl.col('table') == 'housing')

table,r2,rmse,accuracy,f1,roc_auc,f1_weighted
str,f64,f64,f64,f64,f64,f64
"""housing""",null,null,0.883555,0.0,0.67923,null


In [76]:
aug_scores = pl.read_csv('experiments/downstream/logs/simple_results.csv').rename({'': 'experiment'})
aug_scores.filter(pl.col('experiment').str.contains('housing'))

experiment,r2,rmse,accuracy,f1_weighted,f1,roc_auc
str,f64,f64,f64,f64,f64,f64
"""cuk_housing_kitana""",null,null,0.883555,null,0.0,0.67923
"""cuk_housing_backward""",null,null,0.898951,null,0.303694,0.814403
"""cuk_housing_qcr""",null,null,0.896689,null,0.282194,0.802743
"""cuk_housing_arda""",null,null,0.883555,null,0.0,0.67923
"""cuk_housing_forward""",null,null,0.883555,null,0.0,0.67923
"""cuk_housing_autofeat""",null,null,0.883555,null,0.0,0.67923


In [45]:
import polars as pl
import os
import json

base_scores = pl.read_csv('experiments/base_tables/base_simple_scores.csv').rename({'': 'table'})
aug_scores = pl.read_csv('experiments/downstream/logs/simple_results.csv').rename({'_duplicated_0': 'experiment'})
aug_scores = aug_scores.with_columns(pl.col('experiment').fill_null(pl.col('')))

exp_dir = os.listdir('experiments/downstream/logs/')
runtime_res = {}
for exp in exp_dir:
    if not os.path.isdir(f'experiments/downstream/logs/{exp}'):
        continue
    comb_log = f'experiments/downstream/logs/{exp}/{exp}.log'
    with open(comb_log) as f:
        lines = f.readlines()
    for i in range(len(lines)):
        lines[i] = json.loads(lines[i])
    df = pl.from_records(lines, orient='col')
    runtime = df.select(pl.col('runtime').sum()).to_series()[0]
    runtime_res[exp] = runtime
runtime_df = pl.from_dicts([{'experiment': k, 'runtime': v} for k, v in runtime_res.items()])
aug_scores = aug_scores.join(runtime_df, left_on='experiment', right_on='experiment', how='left')
aug_scores = aug_scores.with_columns(
    pl.col('experiment').str.split('_').list.to_struct(upper_bound=3, fields=['lake', 'table', 'algorithm']).struct.unnest()
)
aug_scores = aug_scores.join(base_scores, on='table', how='left', suffix='_base')

aug_scores = aug_scores.with_columns((pl.col('rmse_base').fill_null(0) + pl.col('f1_weighted_base').fill_null(0)).round(3).alias('score_base'))
aug_scores = aug_scores.with_columns((pl.col('rmse').fill_null(0) + pl.col('f1_weighted').fill_null(0)).round(3).alias('score'))
aug_scores = aug_scores.with_columns(
    pl.when(pl.col('rmse').is_not_null())
    .then(pl.lit('rmse'))
    .otherwise(pl.lit('f1'))
    .alias('metric')
)
# Create base rows for each unique (lake, table) combination
base_rows = (
    aug_scores
    .unique(subset=['lake', 'table'])
    .with_columns([
        pl.lit('base').alias('algorithm'),
        pl.col('score_base').alias('score'),
        pl.col('rmse_base').alias('rmse'),
        pl.col('f1_weighted_base').alias('f1'),
        pl.lit(0.0).alias('runtime'),
        (pl.col('lake') + '_' + pl.col('table') + '_base').alias('experiment')
    ])
    .select(aug_scores.columns)
)
aug_scores = pl.concat([aug_scores, base_rows])
aug_scores = aug_scores.with_columns(
    pl.when(pl.col('table') == 'pageviews')
    .then((pl.col('score')/1e6).round(3))
    .otherwise(pl.col('score'))
    .alias('score')
)

In [46]:
# columns_order = ['algorithm', 'airbnb', 'crime', 'flood', 'food', 'housing', 'elections', 'energy', 'flight', 'imdb', 'pageviews', 'vgsales']
algorithm_order = ['base', 'arda', 'autofeat', 'kitana', 'qcr',  'forward', 'backward']
pivoted = (
    aug_scores.with_columns(pl.col('table')+' '+pl.col('metric'))
    .pivot(index='algorithm', columns='table', values='score')
    .with_columns(pl.col('algorithm').cast(pl.Enum(algorithm_order)))
    .sort('algorithm')
)
pivoted

/tmp/ipykernel_3330330/2468167145.py:5: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index='algorithm', columns='table', values='score')


algorithm,realestate rmse,vgsales rmse,trees f1,food f1,fire rmse,hospital f1,energy rmse,pageviews rmse,imdb rmse,arrest f1,jobs rmse,elections rmse
enum,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""base""",806138.849,1.516,0.559,0.597,61659.338,0.258,11.569,5.872,0.524,0.516,5711.347,11.751
"""arda""",806138.849,1.516,0.559,0.597,61659.338,0.258,11.569,5.872,0.524,0.516,5711.347,11.751
"""autofeat""",806138.849,1.516,0.559,0.597,61659.338,0.258,11.569,5.872,0.524,0.516,5711.347,11.751
"""kitana""",806138.849,1.516,0.559,0.597,61659.338,0.258,11.569,5.872,0.524,0.516,5711.347,11.751
"""qcr""",769297.876,1.485,0.559,0.597,61703.165,0.287,9.68,4.51,0.405,0.967,8028.812,11.751
"""forward""",802069.576,1.242,0.64,0.597,61765.732,0.281,9.017,2.965,0.488,0.521,5691.786,11.751
"""backward""",803106.822,1.491,0.559,0.597,61702.083,0.258,8.928,2.834,0.307,0.516,6741.295,11.751


In [8]:
performance = pivoted.to_pandas().to_latex().split('\n')
with open('experiments/downstream/performance.tex', 'w') as f:
    f.write('\n'.join(performance))

In [17]:
algorithm_order = ['base', 'arda', 'autofeat', 'kitana', 'qcr',  'forward', 'backward']
pivoted = (
    aug_scores.with_columns(pl.col('table'))
    .pivot(index='algorithm', columns='table', values='runtime')
    .with_columns(pl.col('algorithm').cast(pl.Enum(algorithm_order)))
    .with_columns(pl.exclude('algorithm').round(3))
    .sort('algorithm')
    .filter(pl.col('algorithm') != 'base')
)
pivoted

/tmp/ipykernel_3288929/4043858860.py:4: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index='algorithm', columns='table', values='runtime')


algorithm,airbnb,vgsales,crime,food,flight,flood,energy,housing,pageviews,imdb,elections
enum,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""arda""",125.28,95.468,179.791,55.653,45.807,54.344,42.833,219.132,56.881,54.171,60.203
"""autofeat""",71.114,68.535,145.65,1225.785,2.416,33.673,1.332,214.165,255.004,74.445,15.409
"""kitana""",0.546,68.158,141.67,1230.005,2.285,34.912,1.433,203.578,253.125,73.814,15.841
"""qcr""",0.129,9.227,0.255,1.104,0.02,0.36,0.316,7.738,1.025,7.862,0.406
"""forward""",286.825,15.712,94.615,33.499,13.64,115.118,66.768,25.615,56.452,29.905,18.223
"""backward""",44.757,13.074,28.923,14.122,4.406,172.193,161.413,15.075,54.897,26.554,14.347


In [20]:
efficiency = pivoted.to_pandas().to_latex()
with open('experiments/downstream/efficiency.tex', 'w') as f:
    f.write(efficiency)

In [146]:
from experiments.base_tables.base_table_preprocessing import PreProcessor
table = 'jobs'
preprocessor = PreProcessor(f'experiments/base_tables/{table}/{table}.csv', f'experiments/base_tables/{table}/splits.json', 0)
X, query_col, target, nan_mask = preprocessor.run()
X.head()

Office Region,cat__Grade,cat__Office Region,Payscale Maximum (Â£)
str,f64,f64,i64
"""southeast""",5.0,5.0,28324
"""southeast""",7.0,5.0,44458
"""london""",9.0,2.0,71674
"""southeast""",5.0,5.0,28324
"""london""",5.0,2.0,28324


In [153]:
cols_to_bin = X.columns[1:-1]

n_bins = 4

binned_exprs = []
for col in cols_to_bin:
    if X[col].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.Int16, pl.Int8, pl.UInt32, pl.UInt64]:
        binned_exprs.append(
            pl.col(col).qcut(n_bins, labels=[str(i) for i in range(n_bins)], allow_duplicates=True).alias(col)
        )
    else:
        binned_exprs.append(pl.col(col))

X = X.select(
    [pl.col(X.columns[0])] + binned_exprs + [pl.col(X.columns[-1])]
)
X.head()

Office Region,cat__Grade,cat__Office Region,Payscale Maximum (Â£)
str,cat,cat,i64
"""southeast""","""0""","""1""",28324
"""southeast""","""1""","""1""",44458
"""london""","""3""","""0""",71674
"""southeast""","""0""","""1""",28324
"""london""","""0""","""0""",28324


In [8]:
df = pl.read_csv('experiments/base_tables/inspections/inspections.csv', ignore_errors=True, separator=',').select(["STREET", "CUISINE DESCRIPTION", "INSPECTION DATE", "ACTION", "CRITICAL FLAG"])
print(df.shape)
df.head()
# df.write_csv('experiments/base_tables/trees/trees.csv', separator=',')

(399918, 5)


STREET,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,CRITICAL FLAG
str,str,str,str,str
"""SECOND AVENUE""","""Italian""","""06/15/2015""","""Violations were cited in the f…","""Critical"""
"""SECOND AVENUE""","""Italian""","""11/25/2014""","""Violations were cited in the f…","""Not Critical"""
"""BROADWAY""","""Italian""","""10/03/2016""","""Violations were cited in the f…","""Not Critical"""
"""HOLDEN BLVD""","""Chinese""","""05/17/2017""","""Violations were cited in the f…","""Critical"""
"""5 AVENUE""","""American""","""03/30/2017""","""Violations were cited in the f…","""Critical"""


In [9]:
', '.join([f'"{col}"' for col in df.columns])

'"STREET", "CUISINE DESCRIPTION", "INSPECTION DATE", "ACTION", "CRITICAL FLAG"'

In [10]:
df.null_count()

STREET,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,CRITICAL FLAG
u32,u32,u32,u32,u32
0,0,0,1135,0


In [11]:
df.select(pl.all().n_unique())

STREET,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,CRITICAL FLAG
u32,u32,u32,u32,u32
3329,84,1414,6,3


In [4]:
from augmentation.retrieval import AurumJoinDiscovery

aurum = AurumJoinDiscovery(index_file='augmentation/Aurum/graphs/nyc.bin')
aurum.find_joinable_tables('/home/fedor/Fast_Data_Discovery/experiments/base_tables/fire/fire.csv_aurum_query.csv', features=['zip_code']).sort('weight')

From 1062 total data-lake tables, scale down to 1062 tables
Loading pre-built index from augmentation/Aurum/graphs/nyc_index.bin...
--- Index Loading Time: 0.00471806526184082 seconds ---


from_id,to_id,from_column,to_column,weight
str,str,str,str,f64
"""/home/fedor/Fast_Data_Discover…","""g8v5-qeu5.csv""","""zip_code""","""col_0""",0.512688
"""/home/fedor/Fast_Data_Discover…","""dvaj-b7yx.csv""","""zip_code""","""col_1""",0.513061
"""/home/fedor/Fast_Data_Discover…","""wvxf-dwi5.csv""","""zip_code""","""col_3""",0.514436
"""/home/fedor/Fast_Data_Discover…","""yc6c-pk2a.csv""","""zip_code""","""col_0""",0.514454
"""/home/fedor/Fast_Data_Discover…","""tg4x-b46p.csv""","""zip_code""","""col_0""",0.524766
…,…,…,…,…
"""/home/fedor/Fast_Data_Discover…","""bhwu-wuzu.csv""","""zip_code""","""col_1""",0.549297
"""/home/fedor/Fast_Data_Discover…","""bhwu-wuzu.csv""","""zip_code""","""col_0""",0.550828
"""/home/fedor/Fast_Data_Discover…","""tg4x-b46p.csv""","""zip_code""","""col_2""",0.558488


In [3]:
import polars as pl
df = pl.read_csv('/mnt/data1/lakes/nyc/extracted/w9ak-ipjd.csv', ignore_errors=True, separator='\t')
df

bin,commmunity_board,total_construction_floor_area,borough,proposed_no_of_stories,block,state,zip,unmapped_cco_street,existing_height,little_e,review_building_code,plumbing_work_type,proposed_height,in_compliance_with_nycecc,applicant_first_name,exempt_from_nycecc,existing_dwelling_units,house_no,building_type,work_on_floor,existing_stories,city,applicant_last_name,street_name,lot,filing_status,specialinspectionrequirement,sprinkler_work_type,job_filing_number,proposed_dwelling_units,includes_permanent_removal,owner_s_street_name,applicant_license,request_legalization,initial_cost,applicant_professional_title,special_inspection_agency_number,progressinspectionrequirement,filing_representative_first_name,filing_representative_street_name,filing_representative_city,filing_representative_last_name,filing_representative_state,filing_representative_zip
i64,i64,i64,str,i64,i64,str,i64,str,i64,str,i64,bool,i64,bool,str,bool,i64,i64,str,str,i64,str,str,str,i64,str,str,bool,str,i64,str,str,i64,str,i64,str,i64,str,str,str,str,str,str,i64
3063093,301,3542,"""BROOKLYN""",5,2417,"""NY""",11362,"""No""",39,"""No""",2014,true,52,false,"""ROMAN""",true,3,124,"""2 Family""","""CEL, BAS, ROF, 1-4""",3,"""DOUGLASTON""","""SOROKKO""","""SOUTH 2nd STREET""",16,"""Objections""","""Sprinkler Systems""",true,"""B00000143-I1""",2,"""No""","""47-30 244 STREET""",72800,"""No""",49500,"""PE""",null,null,null,null,null,null,null,null
3055872,302,3894,"""BROOKLYN""",3,1958,"""NY""",11361,"""No""",41,"""No""",2014,true,41,false,"""CHRIS""",true,3,54,"""Other""","""Cel, Bas, Osp, 1, 2, 3""",3,"""BAYSIDE""","""SIDERIS""","""Greene Avenue""",19,"""Approved""","""Sprinkler Systems""",true,"""B00000043-I1""",3,"""No""","""217-22 NORTHERN BLVD""",65574,"""No""",56800,"""PE""",3434,"""Energy Code Compliance Inspect…",null,null,null,null,null,null
3063093,301,3542,"""BROOKLYN""",5,2417,"""NY""",11362,"""No""",39,"""No""",2014,true,52,false,"""ROMAN""",true,3,124,"""2 Family""","""CEL, BAS, ROF, 1-4""",3,"""DOUGLASTON""","""SOROKKO""","""SOUTH 2nd STREET""",16,"""Objections""","""Fire-Resistant Penetrations an…",true,"""B00000143-I1""",2,"""No""","""47-30 244 STREET""",72800,"""No""",49500,"""PE""",null,null,null,null,null,null,null,null
1077844,107,5625,"""MANHATTAN""",56,1137,"""NY""",10018,"""No""",595,"""No""",2014,false,595,false,"""YURI""",false,341,165,"""Other""","""22""",56,"""NEW YORK""","""KATZ""","""west 65 street""",7501,"""Permit Entire""","""Sprinkler Systems""",true,"""M00000044-I1""",341,"""No""","""36 WEST 37TH STREET""",62121,"""No""",14000,"""PE""",238,"""Final""","""BALJINDER""","""291 BROADWAY, SUITE""","""NEWYORK""","""KAUR""","""NY""",10007
3063093,301,3542,"""BROOKLYN""",5,2417,"""NY""",11362,"""No""",39,"""No""",2014,true,52,false,"""ROMAN""",true,3,124,"""2 Family""","""CEL, BAS, ROF, 1-4""",3,"""DOUGLASTON""","""SOROKKO""","""SOUTH 2nd STREET""",16,"""Objections""","""Post-Installed Anchors""",true,"""B00000143-I1""",2,"""No""","""47-30 244 STREET""",72800,"""No""",49500,"""PE""",null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3321245,301,0,"""BROOKLYN""",4,2352,"""NY""",11358,"""No""",45,"""No""",2014,true,45,false,"""YOUNG SAM""",true,8,642,"""Other""","""CEL""",4,"""FLUSHING""","""YU""","""DRIGGS AVENUE""",25,"""QA Failed""","""High Pressure Fuel- Gas Piping…",false,"""B00000119-I1""",8,"""No""","""162-19 DEPOT ROAD""",36490,"""No""",16000,"""RA""",5252,null,"""JI YOUNG""","""41-23 MURRAY STREET""","""FLUSHING""","""CHOI""","""NY""",11355
3099712,317,0,"""BROOKLYN""",2,4606,"""NY""",11235,"""No""",30,"""No""",2014,true,30,true,"""DMITRY""",null,1,904,"""Other""","""cel, 1""",2,"""BROOKLYN""","""LEVIN""","""RUTLAND ROAD""",1,"""Approved""","""Fire-Resistant Penetrations an…",false,"""B00000150-I1""",1,"""No""","""28 DOOLEY STREET""",95837,"""No""",5000,"""PE""",1267,"""HVAC insulation and sealing""",null,null,null,null,null,null
1079007,101,0,"""MANHATTAN"